In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error

In [3]:
df = pd.read_csv("cleaned_crmls_sold.csv")

train_df = df[df['is_test'] == 0].copy()
test_df = df[df['is_test'] == 1].copy()

feature_cols = [col for col in df.columns if col not in ['target_log_price', 'is_test']]

X_train = train_df[feature_cols]
y_train = train_df['target_log_price']

X_test = test_df[feature_cols]
y_test = test_df['target_log_price']

y_test_original = np.expm1(y_test)

In [5]:
models = {
    "Baseline (Linear Regression)": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest Regressor": RandomForestRegressor(random_state=42, n_estimators=100, max_depth=15, n_jobs=-1)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred_log = model.predict(X_test)
    r2 = r2_score(y_test, y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_log))
    
    y_pred_original = np.expm1(y_pred_log)
    mae_dollar = mean_absolute_error(y_test_original, y_pred_original)
    medae_dollar = median_absolute_error(y_test_original, y_pred_original)
    
    results.append({
        "Model": name,
        "Test R² Score": round(r2, 4),
        "RMSE (Log)": round(rmse, 4),
        "MAE ($)": f"${mae_dollar:,.2f}",
        "Median AE ($)": f"${medae_dollar:,.2f}"
    })

results_df = pd.DataFrame(results)
print("=" * 70)
print("📊 Week 5 Mpdel result (vs Baseline)")
print("=" * 70)
print(results_df.to_string(index=False))

📊 Week 5 Mpdel result (vs Baseline)
                       Model  Test R² Score  RMSE (Log)     MAE ($) Median AE ($)
Baseline (Linear Regression)         0.3535      0.5499 $906,764.03   $343,557.35
     Decision Tree Regressor         0.3488      0.5519 $638,530.70   $327,563.68
     Random Forest Regressor         0.4068      0.5268 $535,075.56   $306,759.44
['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres', 'PropertyType_Residential', 'PropertySubType_SingleFamilyResidence']


In [6]:
rf_model = models["Random Forest Regressor"]
importances = rf_model.feature_importances_

feature_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_imp.head(10).to_string(index=False))

                              Feature  Importance
                           LivingArea    0.410533
                BathroomsTotalInteger    0.314796
                         LotSizeAcres    0.235436
                        BedroomsTotal    0.039235
             PropertyType_Residential    0.000000
PropertySubType_SingleFamilyResidence    0.000000
